# LeetCode 4046 — Minimum Cost Path With At Most K Turns (Hard)

Bài toán LeetCode:

```python
class Solution:
    def minCost(self, grid: List[List[int]], k: int) -> int:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4046 (Weekly Contest 518 — câu 4, rating khoảng 2044).

Bài này luyện cách **mở rộng trạng thái** cho bài đường đi ngắn nhất trên lưới:

1. Trạng thái `(ô, hướng, số lần rẽ)` + **Bellman-Ford** (lặp relax tới khi ổn định) — dễ hiểu, chậm.
2. Cùng trạng thái đó + **Dijkstra** — chuẩn mực.
3. Nhìn lại cấu trúc: đường có `t` lần rẽ = `t + 1` **đoạn thẳng** → **DP theo tầng**, mỗi tầng thêm một đoạn thẳng bằng 4 lượt quét "min đang chạy". Gọn và nhanh nhất.

---

## Sau notebook này, bạn sẽ nắm được

1. Định nghĩa "rẽ" và cách đếm.
2. Mở rộng trạng thái để đưa ràng buộc vào bài đường đi ngắn nhất.
3. Bellman-Ford và Dijkstra trên đồ thị trạng thái.
4. DP theo tầng "đoạn thẳng" với kỹ thuật quét `mn = min(mn + cost, dp)`.
5. Lỗi hay gặp, mẹo nhận diện, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Cho lưới `grid` kích thước `m × n` gồm số nguyên không âm và số `k`. Xuất phát ở ô `(0, 0)`, cần tới ô `(m-1, n-1)`.

- Mỗi bước đi sang một ô kề cạnh (**4 hướng**: lên, xuống, trái, phải).
- Được đi lại ô cũ; **mỗi lần vào ô** (kể cả ô xuất phát) đều phải trả `grid[i][j]`.
- Một **lần rẽ** là khi hướng của bước hiện tại **khác** hướng của bước trước (quay đầu 180° cũng tính là rẽ). Bước đầu tiên không tính là rẽ.
- Được rẽ **tối đa `k` lần**.

Trả về tổng chi phí nhỏ nhất, hoặc `-1` nếu không thể.

---

## Ví dụ 1

```python
grid = [[2, 7, 3],
        [1, 4, 5]], k = 1   ->   12
```

Đi xuống `2 → 1`, rẽ một lần, đi phải `→ 4 → 5`: `2 + 1 + 4 + 5 = 12`.

## Ví dụ 2

```python
grid = [[4, 1, 9],
        [3, 2, 5],
        [4, 8, 6]], k = 2   ->   20
```

`4 ↓ 3 → 2 → 5 ↓ 6`: hướng `xuống, phải, phải, xuống` → rẽ 2 lần; tổng `4 + 3 + 2 + 5 + 6 = 20`.

## Ví dụ 3

```python
grid = [[1, 9],
        [3, 4]], k = 0   ->   -1
```

Không rẽ lần nào thì chỉ đi được theo một đường thẳng — không tới được góc đối diện.

---

## Ràng buộc

```text
1 <= m, n <= 75
0 <= grid[i][j] <= 1000
0 <= k < min(m, n)
```

# 2. Kiến thức nền tảng

## 2.1. Bốn hướng và đếm số lần rẽ

Mã hóa hướng bằng chỉ số `0..3` trong `DIRS`. Một đường đi là dãy hướng; số lần rẽ = số vị trí mà hướng đổi so với bước trước.

In [1]:
from typing import List

DIRS = [(0, 1), (1, 0), (0, -1), (-1, 0)]      # phải, xuống, trái, lên
NAMES = ["phải", "xuống", "trái", "lên"]


def count_turns(dirs: List[int]) -> int:
    return sum(1 for a, b in zip(dirs, dirs[1:]) if a != b)


print(count_turns([1, 0, 0]), count_turns([1, 0, 0, 1]), count_turns([0, 2]))   # [0, 2] = phải rồi trái (quay đầu)

1 2 1


## 2.2. Mở rộng trạng thái

Dijkstra thường chỉ cần trạng thái là **ô**. Nhưng ở đây, việc đi tiếp có rẽ hay không phụ thuộc **hướng vừa đi**, và còn được rẽ bao nhiêu lần phụ thuộc **số lần đã rẽ**. Vì vậy trạng thái là bộ ba:

```text
(ô (i, j), hướng vừa đi d, số lần đã rẽ t)
```

Số trạng thái: `m · n · 4 · (k + 1)` — tối đa `75 · 75 · 4 · 75 ≈ 1.7 triệu`.

## 2.3. Kỹ thuật "min đang chạy" trên một hàng

Cho mảng `best` (chi phí tốt nhất đã có ở mỗi ô, đứng yên) và mảng `cost`. Nếu được đi **thẳng sang phải** bao nhiêu bước tùy ý, chi phí tốt nhất tại ô `j` là

```text
mn_j = min(best[j], mn_{j-1} + cost[j])
```

Một lượt quét `O(n)`.

In [2]:
INF = float("inf")
best = [5, INF, INF, 1, INF]
cost = [5, 2, 3, 1, 4]
mn = INF
out = []
for j in range(len(cost)):
    mn = min(mn + cost[j], best[j])
    out.append(mn)
print(out)

[5, 7, 10, 1, 5]


Ô 1, 2: đi tiếp từ ô 0 (`5 + 2`, `5 + 2 + 3`); ô 3: tự nó tốt hơn (`1`); ô 4: `1 + 4 = 5`.

# 3. Cách 1 — Bellman-Ford trên trạng thái

Lặp: với mọi trạng thái, thử mọi bước đi, cập nhật nếu rẻ hơn. Dừng khi một vòng không còn gì thay đổi. Đơn giản nhưng mỗi vòng `O(số trạng thái × 4)` và có thể cần nhiều vòng.

In [3]:
def min_cost_bellman(grid: List[List[int]], k: int) -> int:
    m, n = len(grid), len(grid[0])
    if m == 1 and n == 1:
        return grid[0][0]
    # best[(i, j, d, t)] ; trạng thái xuất phát dùng d = -1 (chưa có hướng)
    best = {(0, 0, -1, 0): grid[0][0]}
    changed = True
    while changed:
        changed = False
        for (i, j, d, t), c in list(best.items()):
            for nd, (di, dj) in enumerate(DIRS):
                ni, nj = i + di, j + dj
                if not (0 <= ni < m and 0 <= nj < n):
                    continue
                nt = t + (1 if d != -1 and nd != d else 0)
                if nt > k:
                    continue
                nc = c + grid[ni][nj]
                if nc < best.get((ni, nj, nd, nt), INF):
                    best[(ni, nj, nd, nt)] = nc
                    changed = True
    ans = min((c for (i, j, d, t), c in best.items() if (i, j) == (m - 1, n - 1)), default=INF)
    return -1 if ans == INF else ans


print(min_cost_bellman([[2, 7, 3], [1, 4, 5]], 1),
      min_cost_bellman([[4, 1, 9], [3, 2, 5], [4, 8, 6]], 2),
      min_cost_bellman([[1, 9], [3, 4]], 0))

12 20 -1


# 4. Cách 2 — Dijkstra trên trạng thái

Chi phí không âm → Dijkstra: luôn mở rộng trạng thái rẻ nhất; lần đầu lấy ra một trạng thái ở ô đích chính là đáp án.

In [4]:
import heapq


def min_cost_dijkstra(grid: List[List[int]], k: int) -> int:
    m, n = len(grid), len(grid[0])
    best = {}
    pq = [(grid[0][0], 0, 0, -1, 0)]          # (chi phí, i, j, hướng, số lần rẽ)
    while pq:
        c, i, j, d, t = heapq.heappop(pq)
        if (i, j) == (m - 1, n - 1):
            return c
        if best.get((i, j, d, t), INF) < c:
            continue
        for nd, (di, dj) in enumerate(DIRS):
            ni, nj = i + di, j + dj
            if 0 <= ni < m and 0 <= nj < n:
                nt = t + (1 if d != -1 and nd != d else 0)
                if nt > k:
                    continue
                nc = c + grid[ni][nj]
                if nc < best.get((ni, nj, nd, nt), INF):
                    best[(ni, nj, nd, nt)] = nc
                    heapq.heappush(pq, (nc, ni, nj, nd, nt))
    return -1


print(min_cost_dijkstra([[2, 7, 3], [1, 4, 5]], 1),
      min_cost_dijkstra([[4, 1, 9], [3, 2, 5], [4, 8, 6]], 2),
      min_cost_dijkstra([[1, 9], [3, 4]], 0))

12 20 -1


Dijkstra chạy `O(S log S)` với `S ≈ 1.7` triệu trạng thái — đúng nhưng nặng với Python.

---

# 5. Cách 3 — DP theo tầng "đoạn thẳng"

## 5.1. Quan sát

Một đường đi có `t` lần rẽ gồm đúng `t + 1` **đoạn thẳng** (mỗi đoạn đi một hướng cố định). "Rẽ tối đa `k` lần" ⇔ "tối đa `k + 1` đoạn thẳng".

Đặt

```text
dp_s[i][j] = chi phí nhỏ nhất để tới (i, j) bằng tối đa s đoạn thẳng
```

- `dp_0`: chỉ ô xuất phát, `dp_0[0][0] = grid[0][0]`, còn lại `∞`.
- `dp_{s+1}[i][j]` = `min(dp_s[i][j], ` chi phí tới `(i, j)` bằng cách đứng ở một ô `(i', j')` cùng hàng/cột rồi đi **thẳng** tới `(i, j)` `)`.

Điểm hay: khi bắt đầu một đoạn mới ta **không cần biết** đoạn trước đi hướng nào — vì đoạn mới luôn được tính là một đoạn (nếu nó cùng hướng đoạn trước thì ta chỉ dùng **thừa** một đoạn, không sai). Vậy trạng thái chỉ còn là **ô**!

## 5.2. Một tầng = 4 lượt quét

Đi thẳng sang phải trên hàng `i`: dùng "min đang chạy" (mục 2.3) từ trái sang phải. Tương tự phải→trái, trên→dưới, dưới→trên. Mỗi tầng `O(4 · m · n)`, có `k + 1` tầng → `O(k · m · n)`.

## 5.3. Mô phỏng ví dụ 1 (`k = 1` → 2 tầng)

| tầng | dp (∞ viết là `.`) |
|---:|---|
| 0 | `[[2, ., .], [., ., .]]` |
| 1 (≤ 1 đoạn) | `[[2, 9, 12], [3, ., .]]` |
| 2 (≤ 2 đoạn) | `[[2, 9, 12], [3, 7, 12]]` |

Ô đích `(1, 2)` = 12 ✔ (đường `2 ↓ 1 → 4 → 5`; đường kia `2 → 7 → 3 ↓ 5` tốn `17`).

In [5]:
def min_cost_layers(grid: List[List[int]], k: int, verbose: bool = False) -> int:
    m, n = len(grid), len(grid[0])
    dp = [[INF] * n for _ in range(m)]
    dp[0][0] = grid[0][0]
    if verbose:
        show(dp, 0)
    for layer in range(1, k + 2):
        nd = [row[:] for row in dp]
        for i in range(m):
            mn = INF
            for j in range(n):                      # đi sang phải
                mn = min(mn + grid[i][j], dp[i][j])
                nd[i][j] = min(nd[i][j], mn)
            mn = INF
            for j in range(n - 1, -1, -1):          # đi sang trái
                mn = min(mn + grid[i][j], dp[i][j])
                nd[i][j] = min(nd[i][j], mn)
        for j in range(n):
            mn = INF
            for i in range(m):                      # đi xuống
                mn = min(mn + grid[i][j], dp[i][j])
                nd[i][j] = min(nd[i][j], mn)
            mn = INF
            for i in range(m - 1, -1, -1):          # đi lên
                mn = min(mn + grid[i][j], dp[i][j])
                nd[i][j] = min(nd[i][j], mn)
        dp = nd
        if verbose:
            show(dp, layer)
    return -1 if dp[m - 1][n - 1] == INF else dp[m - 1][n - 1]


def show(dp, layer):
    print(f"tầng {layer}:", [[("." if x == INF else x) for x in row] for row in dp])


print(min_cost_layers([[2, 7, 3], [1, 4, 5]], 1, verbose=True))

tầng 0: [[2, '.', '.'], ['.', '.', '.']]
tầng 1: [[2, 9, 12], [3, '.', '.']]
tầng 2: [[2, 9, 12], [3, 7, 12]]
12


Lưu ý: mỗi lượt quét dùng `dp` (tầng **cũ**) làm điểm xuất phát, ghi vào `nd`. Nếu quét thẳng trên `nd` thì một "tầng" có thể chứa **hai** đoạn (ví dụ quét ngang xong rồi quét dọc tiếp từ kết quả ngang) — đếm thiếu số lần rẽ (xem Lỗi 3).

## 5.4. Ví dụ 2 theo tầng

In [6]:
print(min_cost_layers([[4, 1, 9], [3, 2, 5], [4, 8, 6]], 2, verbose=True))

tầng 0: [[4, '.', '.'], ['.', '.', '.'], ['.', '.', '.']]
tầng 1: [[4, 5, 14], [7, '.', '.'], [11, '.', '.']]
tầng 2: [[4, 5, 14], [7, 7, 14], [11, 15, 25]]
tầng 3: [[4, 5, 14], [7, 7, 12], [11, 15, 20]]
20


Tầng 3 (≤ 3 đoạn = ≤ 2 lần rẽ) cho ô đích 20 — đúng đường `4 ↓ 3 → 2 → 5 ↓ 6`.

# 6. Phiên bản đúng format LeetCode

In [7]:
from typing import List


class Solution:
    def minCost(self, grid: List[List[int]], k: int) -> int:
        m, n = len(grid), len(grid[0])
        INF = float("inf")
        dp = [[INF] * n for _ in range(m)]
        dp[0][0] = grid[0][0]
        for _ in range(k + 1):
            nd = [row[:] for row in dp]
            for i in range(m):
                row, g = dp[i], grid[i]
                mn = INF
                for j in range(n):
                    mn = min(mn + g[j], row[j])
                    if mn < nd[i][j]:
                        nd[i][j] = mn
                mn = INF
                for j in range(n - 1, -1, -1):
                    mn = min(mn + g[j], row[j])
                    if mn < nd[i][j]:
                        nd[i][j] = mn
            for j in range(n):
                mn = INF
                for i in range(m):
                    mn = min(mn + grid[i][j], dp[i][j])
                    if mn < nd[i][j]:
                        nd[i][j] = mn
                mn = INF
                for i in range(m - 1, -1, -1):
                    mn = min(mn + grid[i][j], dp[i][j])
                    if mn < nd[i][j]:
                        nd[i][j] = mn
            dp = nd
        return -1 if dp[m - 1][n - 1] == INF else dp[m - 1][n - 1]

In [8]:
sol = Solution()
tests = [
    ([[2, 7, 3], [1, 4, 5]], 1, 12, "ví dụ 1"),
    ([[4, 1, 9], [3, 2, 5], [4, 8, 6]], 2, 20, "ví dụ 2"),
    ([[1, 9], [3, 4]], 0, -1, "ví dụ 3"),
    ([[7]], 0, 7, "lưới 1x1"),
    ([[1, 2, 3, 4]], 0, 10, "một hàng, đi thẳng"),
    ([[0, 0], [0, 0]], 1, 0, "toàn 0"),
]
for grid, k, expected, name in tests:
    result = sol.minCost(grid, k)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:20} -> {result}")

PASS | ví dụ 1              -> 12
PASS | ví dụ 2              -> 20
PASS | ví dụ 3              -> -1
PASS | lưới 1x1             -> 7
PASS | một hàng, đi thẳng   -> 10
PASS | toàn 0               -> 0


# 7. Vì sao đúng?

1. **Tương đương đoạn thẳng:** đường có `t` lần rẽ ⇔ chia được thành `t + 1` đoạn thẳng liên tiếp (cắt tại mỗi chỗ rẽ). Ngược lại, ghép `s` đoạn thẳng bất kỳ (đầu đoạn sau = cuối đoạn trước) cho đường có **tối đa** `s - 1` lần rẽ.
2. **Không cần nhớ hướng:** `dp_s` chỉ cần ô cuối, vì đoạn tiếp theo được trả "một đoạn" bất kể hướng.
3. **Một tầng đúng một đoạn:** mọi lượt quét của tầng `s+1` xuất phát từ `dp_s` (không từ `nd`), và mỗi lượt quét chỉ đi một hướng.
4. **Đi lại ô cũ:** một đoạn thẳng không bao giờ đi qua cùng ô hai lần, còn các đoạn khác nhau có thể đi lại ô cũ — và đều bị tính phí đúng như đề.

# 8. Độ phức tạp

| Cách | Ý tưởng | Thời gian | Bộ nhớ |
|---|---|---:|---:|
| 1. Bellman-Ford | relax lặp trên `(ô, hướng, rẽ)` | `O(V · S)` tệ nhất | `O(S)` |
| 2. Dijkstra | hàng đợi ưu tiên trên trạng thái | `O(S log S)` | `O(S)` |
| 3. DP theo tầng | 4 lượt quét mỗi tầng | `O(k · m · n)` | `O(m · n)` |

(`S = 4 · m · n · (k + 1)`.)

In [9]:
import random
import time

random.seed(4046)
big = [[random.randint(0, 1000) for _ in range(75)] for _ in range(75)]
start = time.time()
print(sol.minCost(big, 74), f"(DP theo tầng: {time.time() - start:.2f} giây)")

42377 (DP theo tầng: 0.22 giây)


# 9. Đối chiếu ba cách

In [10]:
random.seed(46)
for _ in range(600):
    m, n = random.randint(1, 5), random.randint(1, 5)
    grid = [[random.randint(0, 9) for _ in range(n)] for _ in range(m)]
    k = random.randint(0, min(m, n) - 1)
    expected = min_cost_dijkstra(grid, k)
    assert min_cost_bellman(grid, k) == expected, (grid, k)
    assert min_cost_layers(grid, k) == expected, (grid, k)
    assert sol.minCost(grid, k) == expected, (grid, k)
print("Ba cách khớp nhau trên 600 test")

Ba cách khớp nhau trên 600 test


# 10. Những lỗi dễ mắc

## Lỗi 1 — Chỉ cho đi phải / xuống

Thói quen từ *Minimum Path Sum*. Ở đây đi được 4 hướng và có thể đi vòng (đi lên/sang trái để tránh ô đắt). Một đường vòng như `xuống, phải, lên, phải, xuống` cần ít nhất 4 lần rẽ, nên lỗi này chỉ lộ ra ở lưới từ `5 × 5` trở lên — test nhỏ sẽ không bắt được!

In [11]:
def wrong_right_down_only(grid, k):
    m, n = len(grid), len(grid[0])
    from functools import lru_cache

    @lru_cache(None)
    def go(i, j, d, t):
        if t > k:
            return INF
        if (i, j) == (m - 1, n - 1):
            return grid[i][j]
        best = INF
        for nd, (di, dj) in [(0, (0, 1)), (1, (1, 0))]:
            ni, nj = i + di, j + dj
            if ni < m and nj < n:
                best = min(best, grid[i][j] + go(ni, nj, nd, t + (d != -1 and nd != d)))
        return best

    r = go(0, 0, -1, 0)
    return -1 if r == INF else r


grid = [[50, 1, 50, 0, 50],
        [1, 50, 1, 50, 50],
        [1, 50, 50, 0, 1],
        [50, 50, 1, 0, 1],
        [0, 1, 1, 50, 50]]
print("Chỉ phải/xuống:", wrong_right_down_only(grid, 4), "| 4 hướng (đúng):", sol.minCost(grid, 4))

Chỉ phải/xuống: 202 | 4 hướng (đúng): 156


## Lỗi 2 — Tính bước đầu tiên là một lần rẽ

Hướng "ban đầu" chưa có; nếu gán hướng mặc định (ví dụ "phải") thì đi xuống ngay bước đầu bị tính rẽ oan.

In [12]:
def wrong_initial_direction(grid, k):
    m, n = len(grid), len(grid[0])
    best = {}
    pq = [(grid[0][0], 0, 0, 0, 0)]               # SAI: hướng ban đầu = 0 ("phải")
    while pq:
        c, i, j, d, t = heapq.heappop(pq)
        if (i, j) == (m - 1, n - 1):
            return c
        if best.get((i, j, d, t), INF) < c:
            continue
        for nd, (di, dj) in enumerate(DIRS):
            ni, nj = i + di, j + dj
            if 0 <= ni < m and 0 <= nj < n:
                nt = t + (nd != d)
                if nt <= k and c + grid[ni][nj] < best.get((ni, nj, nd, nt), INF):
                    best[(ni, nj, nd, nt)] = c + grid[ni][nj]
                    heapq.heappush(pq, (c + grid[ni][nj], ni, nj, nd, nt))
    return -1


print("Sai :", wrong_initial_direction([[2, 7, 3], [1, 4, 5]], 1), "| Đúng:", sol.minCost([[2, 7, 3], [1, 4, 5]], 1))

Sai : 17 | Đúng: 12


## Lỗi 3 — Quét trên tầng mới (`nd`) thay vì tầng cũ (`dp`)

Quét dọc dùng kết quả vừa quét ngang của cùng tầng → một tầng chứa tới **hai** đoạn → cho phép rẽ nhiều hơn `k`.

In [13]:
def wrong_inplace_layers(grid, k):
    m, n = len(grid), len(grid[0])
    dp = [[INF] * n for _ in range(m)]
    dp[0][0] = grid[0][0]
    for _ in range(k + 1):
        for i in range(m):
            for rng in (range(n), range(n - 1, -1, -1)):
                mn = INF
                for j in rng:
                    mn = min(mn + grid[i][j], dp[i][j]); dp[i][j] = mn      # SAI: ghi đè ngay
        for j in range(n):
            for rng in (range(m), range(m - 1, -1, -1)):
                mn = INF
                for i in rng:
                    mn = min(mn + grid[i][j], dp[i][j]); dp[i][j] = mn
    return -1 if dp[m - 1][n - 1] == INF else dp[m - 1][n - 1]


print("Sai :", wrong_inplace_layers([[1, 9], [3, 4]], 0), "| Đúng:", sol.minCost([[1, 9], [3, 4]], 0))

Sai : 14 | Đúng: -1


## Lỗi 4 — Không coi quay đầu là rẽ

Đề nói "hướng khác hướng trước" — quay đầu 180° cũng là khác. Trong DP theo tầng, điều này tự động đúng (mỗi đoạn một hướng).

## Lỗi 5 — Quên trả phí ô xuất phát

Đáp án ví dụ 1 là `12 = 2 + 1 + 4 + 5`, đã gồm `grid[0][0] = 2`.

# 11. Mẹo nhận diện

1. Đường đi ngắn nhất có **ràng buộc phụ** (số lần rẽ, số lần dùng phép đặc biệt, nhiên liệu...) → **mở rộng trạng thái** rồi Dijkstra/BFS.
2. Ràng buộc là "số đoạn thẳng" / "số lần đổi hướng" → **DP theo tầng**, mỗi tầng một đoạn; trong một đoạn dùng quét tiền tố.
3. Chi phí không âm, lưới nhỏ (`75 × 75`) nhưng `k` lớn → `O(k · m · n)` là vừa đẹp.

Các bài liên quan: *Minimum Path Sum* (LeetCode 64), *Shortest Path in a Grid with Obstacles Elimination* (1293 — trạng thái thêm số lần phá tường), *Minimum Cost to Make at Least One Valid Path in a Grid* (1368 — 0-1 BFS theo hướng).

# 12. Biến thể

## Biến thể 1 — Đúng `k` lần rẽ

Phải dùng trạng thái có hướng (không gộp được như DP tầng), vì "đoạn thừa cùng hướng" không còn vô hại.

## Biến thể 2 — Ít lần rẽ nhất (bỏ qua chi phí), có ô cấm

BFS theo tầng đoạn thẳng: tầng `s` = các ô tới được bằng `s` đoạn. (Bài tập 5.)

# 13. Template tư duy

```python
# DP theo tầng đoạn thẳng
dp = [[INF] * n for _ in range(m)]; dp[0][0] = grid[0][0]
for _ in range(k + 1):                       # mỗi tầng thêm 1 đoạn thẳng
    nd = copy(dp)
    for mỗi hàng/cột, mỗi chiều quét:
        mn = INF
        for ô theo chiều quét:
            mn = min(mn + grid[ô], dp[ô])    # đi tiếp hoặc bắt đầu đoạn tại đây
            nd[ô] = min(nd[ô], mn)
    dp = nd
```

# 14. Bài tập vận dụng

## Bài 1 — Tính bằng tay

`grid = [[1, 5, 1], [2, 9, 1], [8, 1, 1]]`, `k = 1`. Tính tầng 1 và tầng 2 của DP. Kiểm tra bằng cell dưới.

In [14]:
# Kiểm tra Bài 1
print(min_cost_layers([[1, 5, 1], [2, 9, 1], [8, 1, 1]], 1, verbose=True))

tầng 0: [[1, '.', '.'], ['.', '.', '.'], ['.', '.', '.']]
tầng 1: [[1, 6, 7], [3, '.', '.'], [11, '.', '.']]
tầng 2: [[1, 6, 7], [3, 12, 8], [11, 12, 9]]
9


## Bài 2 — Dự đoán output

```python
print(sol.minCost([[1, 1, 1], [1, 1, 1], [1, 1, 1]], 1))
print(sol.minCost([[1, 1, 1], [1, 1, 1], [1, 1, 1]], 2))
print(count_turns([0, 1, 0, 1]))
```

In [15]:
print(sol.minCost([[1, 1, 1], [1, 1, 1], [1, 1, 1]], 1))
print(sol.minCost([[1, 1, 1], [1, 1, 1], [1, 1, 1]], 2))
print(count_turns([0, 1, 0, 1]))

5
5
3


## Bài 3 — Quét một chiều

Viết hàm nhận `best` và `cost` (hai list cùng độ dài), trả về chi phí tốt nhất tại mỗi ô nếu được đi thẳng **sang trái** (từ phải qua).

```python
sweep_left([INF, INF, 0], [3, 2, 5])   -> [5, 2, 0]
```

In [16]:
def exercise_3_sweep_left(best: List[float], cost: List[int]) -> List[float]:
    # TODO
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_sweep_left([INF, INF, 0], [3, 2, 5]) == [5, 2, 0]

## Bài 4 — Minimum Path Sum (LeetCode 64)

Chỉ đi phải/xuống, tổng nhỏ nhất từ góc trên trái tới góc dưới phải.

```python
min_path_sum([[1, 3, 1], [1, 5, 1], [4, 2, 1]])   -> 7
```

In [17]:
def exercise_4_min_path_sum(grid: List[List[int]]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_4_min_path_sum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]) == 7

## Bài 5 — Ít lần rẽ nhất, có ô cấm

Lưới gồm `'.'` (đi được) và `'#'` (cấm). Tìm số lần rẽ ít nhất để đi từ `(0, 0)` tới `(m-1, n-1)`, hoặc `-1`.

```python
min_turns(["..#", "#..", "..."])   -> 2     # phải, xuống, phải, xuống
```

In [18]:
def exercise_5_min_turns(board: List[str]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_5_min_turns(["..#", "#..", "..."]) == 2

## Bài 6 — Mini challenge: chỉ đi phải/xuống, tối đa `k` lần rẽ

Chi phí nhỏ nhất khi chỉ được đi **phải hoặc xuống** và rẽ tối đa `k` lần. Gợi ý: `dp[i][j][d][t]`.

In [19]:
def exercise_6_right_down_k(grid: List[List[int]], k: int) -> int:
    # TODO
    raise NotImplementedError

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: duyệt `j` từ `n-1` về `0`.
- Mức 2: `mn = min(mn + cost[j], best[j])`.
- Mức 3: ghi `out[j] = mn`.

**Bài 4**

- Mức 1: `dp[i][j] = grid[i][j] + min(dp[i-1][j], dp[i][j-1])`.
- Mức 2: hàng/cột đầu chỉ có một nguồn.
- Mức 3: trả về `dp[m-1][n-1]`.

**Bài 5**

- Mức 1: tầng `s` = tập ô tới được bằng tối đa `s` đoạn thẳng.
- Mức 2: từ mỗi ô vừa thêm, đi thẳng 4 hướng tới khi gặp `#`/biên.
- Mức 3: đáp án = (tầng đầu tiên chứa đích) − 1.

**Bài 6**

- Mức 1: `d ∈ {phải, xuống}`; `t` = số lần rẽ.
- Mức 2: từ `(i, j, d, t)` đi tiếp hướng `nd`: `t + (nd != d)`.
- Mức 3: hai bước đầu (phải hoặc xuống từ ô xuất phát) không tính rẽ.

# 15. Đáp án tham khảo

## Đáp án Bài 1

Tầng 1 (≤ 1 đoạn): hàng 0 `[1, 6, 7]`, cột 0 `[1, 3, 11]`.
Tầng 2 (≤ 2 đoạn): ô đích `(2, 2)`: `1 → 5 → 1 ↓ 1 ↓ 1 = 9` hoặc `1 ↓ 2 ↓ 8 → 1 → 1 = 13` → **9**. (So khớp với bảng in ra.)

## Đáp án Bài 2

```text
5
5
3
```

Lưới toàn 1: mọi đường ngắn nhất qua 5 ô; với `k = 1` đã đi được `→ → ↓ ↓`. `count_turns([0, 1, 0, 1]) = 3`.

In [20]:
# Đáp án Bài 3
def answer_3_sweep_left(best: List[float], cost: List[int]) -> List[float]:
    out = [INF] * len(cost)
    mn = INF
    for j in range(len(cost) - 1, -1, -1):
        mn = min(mn + cost[j], best[j])
        out[j] = mn
    return out


assert answer_3_sweep_left([INF, INF, 0], [3, 2, 5]) == [5, 2, 0]
print("Bài 3: PASS")

Bài 3: PASS


In [21]:
# Đáp án Bài 4
def answer_4_min_path_sum(grid: List[List[int]]) -> int:
    m, n = len(grid), len(grid[0])
    dp = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            if i == 0 and j == 0:
                dp[i][j] = grid[0][0]
            else:
                dp[i][j] = grid[i][j] + min(dp[i - 1][j] if i else INF, dp[i][j - 1] if j else INF)
    return dp[m - 1][n - 1]


assert answer_4_min_path_sum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]) == 7
print("Bài 4: PASS")

Bài 4: PASS


In [22]:
# Đáp án Bài 5
def answer_5_min_turns(board: List[str]) -> int:
    m, n = len(board), len(board[0])
    if board[0][0] == "#" or board[m - 1][n - 1] == "#":
        return -1
    seen = {(0, 0)}
    frontier = [(0, 0)]
    segments = 0
    while frontier:
        if (m - 1, n - 1) in seen:
            return max(segments - 1, 0)
        segments += 1
        nxt = []
        for i, j in frontier:
            for di, dj in DIRS:
                ni, nj = i + di, j + dj
                while 0 <= ni < m and 0 <= nj < n and board[ni][nj] == ".":
                    if (ni, nj) not in seen:
                        seen.add((ni, nj))
                        nxt.append((ni, nj))
                    ni, nj = ni + di, nj + dj
        frontier = nxt
    return max(segments - 1, 0) if (m - 1, n - 1) in seen else -1


assert answer_5_min_turns(["..#", "#..", "..."]) == 2
assert answer_5_min_turns(["...", "...", "..."]) == 1
assert answer_5_min_turns([".#", "#."]) == -1
assert answer_5_min_turns(["."]) == 0
print("Bài 5: PASS")

Bài 5: PASS


In [23]:
# Đáp án Bài 6
from functools import lru_cache


def answer_6_right_down_k(grid: List[List[int]], k: int) -> int:
    m, n = len(grid), len(grid[0])

    @lru_cache(None)
    def go(i, j, d, t):              # đang ở (i, j), hướng vừa đi d (-1 = chưa đi), đã rẽ t lần
        if (i, j) == (m - 1, n - 1):
            return grid[i][j]
        best = INF
        for nd, (di, dj) in ((0, (0, 1)), (1, (1, 0))):
            ni, nj = i + di, j + dj
            nt = t + (1 if d != -1 and nd != d else 0)
            if ni < m and nj < n and nt <= k:
                best = min(best, grid[i][j] + go(ni, nj, nd, nt))
        return best

    r = go(0, 0, -1, 0)
    return -1 if r == INF else r


def brute_right_down(grid, k):
    m, n = len(grid), len(grid[0])
    best = INF

    def dfs(i, j, d, t, c):
        nonlocal best
        if t > k:
            return
        if (i, j) == (m - 1, n - 1):
            best = min(best, c)
            return
        if j + 1 < n:
            dfs(i, j + 1, 0, t + (d == 1), c + grid[i][j + 1])
        if i + 1 < m:
            dfs(i + 1, j, 1, t + (d == 0), c + grid[i + 1][j])

    dfs(0, 0, -1, 0, grid[0][0])
    return -1 if best == INF else best


random.seed(6)
for _ in range(500):
    m, n = random.randint(1, 5), random.randint(1, 5)
    g = [[random.randint(0, 9) for _ in range(n)] for _ in range(m)]
    kk = random.randint(0, 3)
    assert answer_6_right_down_k(g, kk) == brute_right_down(g, kk), (g, kk)
print("Bài 6: PASS (khớp brute force)")

Bài 6: PASS (khớp brute force)


# 16. Tổng kết

**Minimum Cost Path With At Most K Turns**:

1. Trạng thái `(ô, hướng, số lần rẽ)` → Bellman-Ford / Dijkstra: đúng nhưng nặng.
2. Quan sát: `t` lần rẽ = `t + 1` đoạn thẳng → DP theo tầng, trạng thái chỉ còn **ô**.
3. Mỗi tầng = 4 lượt quét `mn = min(mn + grid, dp_cũ)`; luôn quét từ tầng cũ.

## Độ phức tạp

```text
Time:  O(k · m · n)
Space: O(m · n)
```

# 17. Checklist trước khi submit

- [ ] Đi được **4 hướng**, được đi lại ô cũ?
- [ ] Bước đầu tiên không tính rẽ; quay đầu tính là rẽ?
- [ ] `k + 1` tầng (đoạn), mỗi lượt quét xuất phát từ `dp` tầng cũ?
- [ ] Đã trả phí ô xuất phát?
- [ ] Đã test: lưới 1×1, `k = 0`, một hàng?